# Context

"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base. One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering - Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information. The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being. However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.

#### Dataset Source: https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("./Travel.xls")
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [2]:
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
%matplotlib inline

 ### Data Cleaning
 #### Handling 
 1) missing values
 2) duplicates
 3) check data type
 4) Understand the dataset

In [3]:
# checking of categories
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [4]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [5]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [6]:
df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64

In [7]:
# Checking of missing values count in each column
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [8]:
df.isnull().mean()*100 # just checking the mean values, optional


CustomerID                  0.000000
ProdTaken                   0.000000
Age                         4.623568
TypeofContact               0.511457
CityTier                    0.000000
DurationOfPitch             5.135025
Occupation                  0.000000
Gender                      0.000000
NumberOfPersonVisiting      0.000000
NumberOfFollowups           0.920622
ProductPitched              0.000000
PreferredPropertyStar       0.531915
MaritalStatus               0.000000
NumberOfTrips               2.864157
Passport                    0.000000
PitchSatisfactionScore      0.000000
OwnCar                      0.000000
NumberOfChildrenVisiting    1.350245
Designation                 0.000000
MonthlyIncome               4.766776
dtype: float64

In [9]:
# fix for missing values
 # take previous function df.is_null().sum() and find where it is having at least 1 null value
# features_with_na = [features for features in df.columns]
# print(features_with_na)
# next add if condition to output only columns with at least 1 null value
features_with_na = [features for features in df.columns if df[features].isnull().sum() >= 1]
print(features_with_na)


['Age', 'TypeofContact', 'DurationOfPitch', 'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips', 'NumberOfChildrenVisiting', 'MonthlyIncome']


In [13]:
# next we want the percentage of missing value
for feature in features_with_na:
    print(feature,np.round(df[feature].isnull().mean()*100,2),'% of missing values')
    # mean = add all values (0+1+0+1/4 total values) and multiply by 100 to get the percentage

Age 4.62 % of missing values
TypeofContact 0.51 % of missing values
DurationOfPitch 5.14 % of missing values
NumberOfFollowups 0.92 % of missing values
PreferredPropertyStar 0.53 % of missing values
NumberOfTrips 2.86 % of missing values
NumberOfChildrenVisiting 1.35 % of missing values
MonthlyIncome 4.77 % of missing values


.select_dtypes(exclude='object')

👉 “From those columns… remove text columns”

Because:

'object' = text (like names, cities)
You keep only numbers

👉 Now you have only numeric columns with missing values

In [34]:
# to check whether my data has some outliers or not
# how we check is we compare "mean" with the 50% percentile value and check if there is a big gap in them. if it isn't then we are good.
# .loc[['mean', '50%']] acts as a filter to keep only the required statistics.
df[features_with_na].select_dtypes(exclude='object').describe().loc[['mean', '50%']]


,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000


### Next we will impute null values. we will replace integer values with median and categorical values with mode.
Age                         226 | Median  
TypeofContact                25 | MODE  
DurationOfPitch             251 | Median  
NumberOfFollowups            45 | MODE as it is a discrete feature  
PreferredPropertyStar        26 | MODE  
NumberOfTrips               140 | Median  
NumberOfChildrenVisiting     66 | MODE  
MonthlyIncome               233 | Median  

In [39]:
# to find median we do 
tempx = df.Age.median()
print(tempx)

36.0


In [49]:
# to find mode we do
tempy =  df.NumberOfFollowups.mode()[0]
# You use df['Type of Contact'].mode()[0] to extract the single most frequent value (the mode) from a pandas Series. The .mode() method returns a Series because a dataset can have multiple modes (a tie), so [0] selects the first one, allowing you to use it for operations like fillna().
print(tempy)

4.0


In [54]:
# Age
df.Age.fillna(df.Age.median(),inplace=True)

# Type of contract
df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)

#DurationOfPitch             251 | Median 
df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True)

# NumberOfFollowups            45 | MODE as it is a discrete feature
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)

# PreferredPropertyStar        26 | MODE 
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)

# NumberOfTrips               140 | Median
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)

# NumberOfChildrenVisiting     66 | MODE  
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)

# MonthlyIncome 
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)

In [56]:
# Once more let us sum the number of null values
df.isnull().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64